# Canonical Model 04: PEST Calibration and Results

This notebook calibrates the **canonical valley model itself**. The model's hydraulic conductivity is reset to a wrong starting value (3x too transmissive); head observations are sampled from the true K field, so there is real work for history matching to do.

We use the declarative `PestProject` facade -- `parameterize`, `observe`, `forecast`, `build` -- which compiles straight to native `pyemu.utils.PstFrom`, run PESTPP-IES, and then review **how well the calibrated ensemble reproduces the observations**. (Notebook 06 takes the same setup further into forecast *uncertainty*.)

In [ ]:
import sys
from pathlib import Path

# Make the in-repo `src/` importable when myflopy is not pip-installed.
src = Path.cwd().parents[2] / "src"
if src.exists() and str(src) not in sys.path:
    sys.path.insert(0, str(src))

import pandas as pd
import myflopy as mf
from canonical_notebook_style import notebook_header
from myflopy.modflow.mf6.canonical_calibration import build_canonical_calibration_demo
from myflopy.modflow.mf6.pest import PestProject

notebook_header('04', 'PEST Calibration and Results',
                'Calibrate the canonical model with PESTPP-IES, then review the fit.')

## 1. Build the calibration demo and declare the PEST problem

`build_canonical_calibration_demo` builds the canonical model as the synthetic truth, samples head observations across the valley floor (plus one down-valley head forecast near the lake), and resets K to the wrong start. We then declare two parameters, the head observations, and the forecast, and build the control file with `noptmax=0` (evaluate once and compute residuals).

In [ ]:
artifact_root = Path('../artifacts/canonical_pest')
artifact_root.mkdir(parents=True, exist_ok=True)

demo = build_canonical_calibration_demo(artifact_root / 'model')

cal = PestProject(model=demo.model, name='canonical_pest',
                  workspace=artifact_root / 'template', start_datetime='2024-01-01')
# One constant K multiplier (scales every layer's K file together) and one
# constant recharge multiplier. Bounds are multiplier factors; `physical`
# clamps the final model value so calibration cannot reach nonphysical K.
cal.parameterize('k',        style='constant', bounds=(0.05, 2.0), physical=(0.01, 300.0))
cal.parameterize('recharge', style='constant', bounds=(0.3, 3.0),  physical=(0.0, 1e-2))
cal.observe(demo.head_targets)
cal.forecast(demo.forecast_targets)
pst = cal.build('canonical_pest.pst', noptmax=0)
print(cal.settings())

In [ ]:
pd.Series({
    'cells': int(demo.model.vor.ncpl),
    'observation_wells': demo.head_targets.locations_gdf.shape[0],
    'head_observations': len(demo.head_targets.to_long()),
    'adjustable_parameters': pst.npar_adj,
    'nonzero_observations': pst.nnz_obs,
    'start_K_factor': demo.start_k_factor,
}, name='calibration problem')

## 2. Validate the forward run

Every PEST iteration calls `forward_run.py`. Running it once directly is the key setup check: it must apply the parameter multipliers, run MODFLOW, and regenerate the simulated-observation files. **What to look for:** a zero return code and a regenerated `hds_simulated_heads.csv`.

In [ ]:
import subprocess
template = cal.template_workspace
result = subprocess.run([sys.executable, 'forward_run.py'], cwd=template,
                        capture_output=True, text=True)
assert result.returncode == 0, result.stdout + '\n' + result.stderr
pd.Series({
    'forward_run_returncode': result.returncode,
    'apply_list_and_array_pars_present': 'apply_list_and_array_pars' in (template / 'forward_run.py').read_text(),
    'simulated_heads_regenerated': (template / 'hds_simulated_heads.csv').exists(),
}, name='forward-run validation')

## 3. Run the calibration (PESTPP-IES)

`run_ies` configures the ensemble options, launches PESTPP-IES with parallel agents, and returns an `IesResults`. The forward model is package-heavy (~7 s per run) and IES fires hundreds of runs, so this is a *go-get-coffee* cell -- use parallel `workers` and a modest ensemble for the demo. Set `RUN_IES = False` to skip it.

In [ ]:
RUN_IES = True       # set False to skip the (slow) ensemble run
REALS = 30
ITERATIONS = 3
WORKERS = 12         # parallel PESTPP-IES agents (this machine has plenty of cores)

if RUN_IES:
    ies = cal.run_ies(reals=REALS, iterations=ITERATIONS, workers=WORKERS)
    print(ies.settings)
else:
    ies = None
    print('Skipped: set RUN_IES = True to run PESTPP-IES.')

## 4. Did the misfit drop? (phi convergence)

The first thing to check: did history matching reduce the objective function (phi)? Each faint line is one realization; the bold line is the ensemble mean. **What to look for:** a clear drop from the prior to the posterior, *without* the ensemble collapsing to a single line (which signals over-fitting).

In [ ]:
if ies is not None:
    display(ies.plot_phi())                       # plotly
    display(ies.plot_phi(backend='matplotlib'))   # seaborn/matplotlib

## 5. Observed versus simulated

Grey is the prior ensemble, blue the posterior, red the measured heads. **What to look for:** the posterior (blue) spread should bracket the red markers -- close enough to fit, but not implausibly narrow.

In [ ]:
if ies is not None:
    display(ies.plot_vs_obs())

## 6. Forecast summary and review bundle

`forecasts()` summarizes prior -> posterior uncertainty for the down-valley head prediction. `report(...)` bundles phi convergence, the ensemble-vs-observation comparison, and the forecast histograms into one self-contained HTML file. (Notebook 06 dissects the forecast uncertainty in depth.)

In [ ]:
if ies is not None:
    display(ies.forecasts())
    report_path = ies.report(artifact_root / 'calibration_review.html')
    print('wrote', report_path)

## Interpretation checklist

- Confirm phi actually dropped and the ensemble did not collapse to a single line.
- Check the posterior brackets the measured heads without being implausibly narrow.
- A lower objective function is *evidence*, not the whole deliverable -- carry forward the **base** realization, never the lowest-phi one.
- For the prediction you care about, value the posterior *spread*, not just a reduced phi -- see **Notebook 06** for the full uncertainty analysis.